# Mountain hazard source-zone screening — Kaggle / Colab runner

Runs the minimum viable pipeline from the paper on real Sentinel-2 data.

**Kaggle settings:** Accelerator *GPU T4*, Internet *On*.
Upload the `mountain-hazard-screening` folder (zipped) as a Kaggle Dataset first, or upload the zip to Colab.

Outputs are screening products for specialist review, not forecasts or warnings.

In [ ]:
# 1. Install
import os, sys, glob
SRC = glob.glob('/kaggle/input/*/mountain-hazard-screening') or glob.glob('/content/mountain-hazard-screening')
assert SRC, 'upload the project folder first'
!cp -r {SRC[0]} /kaggle/working/mhs-project 2>/dev/null || cp -r {SRC[0]} /content/mhs-project
PROJ = '/kaggle/working/mhs-project' if os.path.exists('/kaggle/working') else '/content/mhs-project'
%cd {PROJ}
!pip install -q -e ".[acquire]"

In [ ]:
# 2. Sanity check: synthetic demo (dev catchment -> unseen test catchment), ~1 min on GPU
!mhs demo --out demo_output --epochs 15
from IPython.display import Image, Markdown, display
display(Image('demo_output/run_test/overview.png'))

## Real case: Chamoli 2021
Check `configs/chamoli.yaml` first: bbox, dates (comparable season), and leave `thickness_scenarios_m`
empty until you have published or field thickness evidence.

In [ ]:
# 3. Download Sentinel-2 L2A (3 dates) and Copernicus DEM onto one 10 m grid
!mhs acquire configs/chamoli.yaml --out data/chamoli
!cat data/chamoli/acquisition.json

In [ ]:
# 4. Bootstrap PARTIAL labels (water, snow/ice, vegetation, steep rock only)
!mhs weak-labels configs/chamoli.yaml --out data/chamoli/labels.tif
# Download labels.tif, open it in QGIS over t1/t2, digitise soil and debris (values 4 and 5),
# correct errors, and upload it back to data/chamoli/labels.tif before training.
# Weak labels must never be used as the evaluation reference.

In [ ]:
# 5. Assets: write data/chamoli/assets.geojson (lon/lat) with properties
#    type (settlement / road / bridge / hydropower / utility), name, population, data_date, source.
#    Use OSM, Census or official sources and record their dates.

In [ ]:
# 6. Train the U-Net (east strip held out for testing, seed fixed)
!mhs train configs/chamoli.yaml --out models/chamoli

In [ ]:
# 7. Screening run
!mhs run configs/chamoli.yaml --model models/chamoli/unet.pt --out runs/chamoli
display(Image('runs/chamoli/overview.png'))
display(Markdown(open('runs/chamoli/report.md').read()))

In [ ]:
# 8. Section 9 sensitivity calculation (published values)
!mhs chamoli | head -40

### Recompute the collapse volume from the original DEMs
Download the pre/post 2 m DEM composites from Shean et al. (2021), Zenodo doi:10.5281/zenodo.4558692,
and digitise the source polygon (GeoJSON, lon/lat). Then:

```
!mhs chamoli --pre data/shean/pre.tif --post data/shean/post.tif --polygon data/shean/source.geojson --ref-elev <valley floor m>
```
To use the DEMs inside the pipeline (changed volume per zone), resample them first:
```
!mhs regrid data/shean/pre.tif  --like data/chamoli/t1.tif --out data/chamoli/dem_pre_10m.tif
!mhs regrid data/shean/post.tif --like data/chamoli/t1.tif --out data/chamoli/dem_post_10m.tif
```
and set `dem_t1` / `dem_t2` in the config.

In [ ]:
# 9. Archive the run (config, hashes, model, maps, report) for reproducibility
!cd runs && zip -qr chamoli_run.zip chamoli && ls -lh chamoli_run.zip